# Compare trial curvature

Occupied traced Berry curvature for the two Ti_Q_2A trials (trial03: Sr sp, Ti pd, O sp / 48 WFs; trial04: Sr sp, Ti p, O sp / 38 WFs), evaluated on the generic line $\mathbf{k}(t) = (t, 0.30, 0.15)$. Both Wannierizations describe the same 38-band occupied manifold; the comparison uses the center-aware position matrix reconstructed from each trial's MMN overlaps, with the same orbital-dependent real-space mapping applied to H(R) and A(R).

Set `RUN_COMPUTATION = False` below to skip straight to loading the last saved run from `results/trial_curvature/` and inspecting/plotting it.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import json

REPO_ROOT = Path.cwd()
while not (REPO_ROOT / "validation").exists():
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

from validation.symmetry.compare_trial_curvature_lib import (
    TRIALS,
    OUTPUT_DIR,
    build_trial,
    evaluate_trial,
    comparison_metrics,
    ab_initio_grid_subspace_metrics,
    save_results,
    load_results,
    plot_comparison,
)

## Config

Edit these, then re-run from here. `TRIALS` (imported above) holds each trial's directory/cache/label; override entries directly if you want to point at a different run.

In [ ]:
RUN_COMPUTATION = False  # True: rebuild both trials from the Wannier run and recompute. False: load results/trial_curvature/.
POINTS = 201
OUTPUT_DIR = OUTPUT_DIR  # results/trial_curvature; point elsewhere to compare against a different saved run

## Build trial03

In [ ]:
if RUN_COMPUTATION:
    model03, pos03 = build_trial("trial03", TRIALS)

## Build trial04

In [ ]:
if RUN_COMPUTATION:
    model04, pos04 = build_trial("trial04", TRIALS)

## Evaluate both trials on the generic line

`results[key]["curvature"]` is the full Cartesian-indexed traced-occupied curvature tensor `Omega[i, j, k_index, occ, occ]` -- inspect it directly here.

In [ ]:
if RUN_COMPUTATION:
    t = np.linspace(0.0, 1.0, POINTS)
    kpoints = np.column_stack((t, np.full_like(t, 0.30), np.full_like(t, 0.15)))
    results = {
        "trial03": evaluate_trial(model03, pos03, kpoints),
        "trial04": evaluate_trial(model04, pos04, kpoints),
    }
    results["trial03"]["curvature"].shape

## Metrics: pointwise/norm differences + occupied-subspace overlap

In [ ]:
if RUN_COMPUTATION:
    metrics = comparison_metrics(results)
    metrics["ab_initio_grid_subspace"] = ab_initio_grid_subspace_metrics(TRIALS)
    print(json.dumps(metrics, indent=2, sort_keys=True))

## Save (only runs in compute mode)

In [ ]:
if RUN_COMPUTATION:
    save_results(t, kpoints, results, metrics, OUTPUT_DIR)

## Load previously saved results (skip if you just computed above)

In [ ]:
if not RUN_COMPUTATION:
    loaded, metrics = load_results(OUTPUT_DIR)
    t, kpoints, results = loaded["t"], loaded["kpoints"], loaded["results"]
    print(json.dumps(metrics, indent=2, sort_keys=True))

## Plot

In [ ]:
fig, axes = plot_comparison(t, results, TRIALS)
plt.show()